# Level 2: complex data

Builds telemetry (10 million rows by default), SCD2 equipment, BOM bridge, maintenance work orders, suppliers, lots, and a shift calendar. Set `telemetry_rows` to `50000000` only if you have time and a Medium warehouse.


## Bind config including telemetry size


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("telemetry_rows", "10000000", "Telemetry rows (10000000 or 50000000)")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)


## Gold shift fact plus dimensions

Creates the governed grain `fct_line_shift` that metric views will use. Telemetry is liquid-clustered on line_id and ts.


In [ ]:
from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {cfg.fqn}")
n = cfg.telemetry_rows
assert n in (10_000_000, 50_000_000)

spark.sql(f"CREATE OR REPLACE TABLE {cfg.table('shift_calendar')} AS SELECT explode(sequence(date_sub(current_date(), 90), date_sub(current_date(), 1))) AS shift_date")

spark.sql(f"""
CREATE OR REPLACE TABLE {cfg.table('equipment')} (
  asset_id STRING, line_id STRING, firmware STRING, valid_from DATE, valid_to DATE, is_current BOOLEAN
)
""")
spark.sql(f"""
INSERT OVERWRITE {cfg.table('equipment')}
SELECT concat('AST-', id), concat('LN-0', (id % 6) + 1, '-0', (id % 4) + 1),
       concat('fw-', (id % 3) + 1), date_sub(current_date(), 400), DATE '9999-12-31', true
FROM range(48)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {cfg.table('bom')} AS SELECT id AS bom_id, concat('FG-', id) AS finished_good FROM range(12)
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {cfg.table('bom_component_bridge')} AS
SELECT b.bom_id, concat('CMP-', (b.bom_id + x.id) % 20) AS component_id, 1 + (x.id % 3) AS qty
FROM {cfg.table('bom')} b CROSS JOIN range(4) x
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {cfg.table('suppliers')} AS
SELECT concat('SUP-', id) AS supplier_id, concat('Supplier ', id) AS supplier_name FROM range(8)
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {cfg.table('component_lots')} AS
SELECT concat('LOT-', id) AS lot_id, concat('CMP-', id % 20) AS component_id, concat('SUP-', id % 8) AS supplier_id,
       date_sub(current_date(), cast(id % 60 AS INT)) AS received_date
FROM range(500)
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {cfg.table('maintenance_work_orders')} AS
SELECT concat('WO-', id) AS wo_id,
       concat('LN-0', (id % 6) + 1, '-0', (id % 4) + 1) AS line_id,
       CASE WHEN id % 4 = 0 THEN 'unplanned' ELSE 'planned' END AS wo_type,
       date_sub(current_date(), cast(id % 80 AS INT)) AS opened_date,
       15 + (id % 240) AS mttr_minutes
FROM range(800)
""")

telemetry = (
    spark.range(n)
    .withColumn("line_id", F.concat(F.lit("LN-0"), (F.col("id") % 6 + 1).cast("string"), F.lit("-0"), (F.col("id") % 4 + 1).cast("string")))
    .withColumn("ts", F.expr("timestampadd(SECOND, id % 86400, to_timestamp(date_sub(current_date(), 1)))"))
    .withColumn("reading", F.struct((F.rand(42) * 100).alias("temp_c"), (F.rand(7) * 20).alias("vibration")))
)
(
    telemetry.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(cfg.table("sensor_telemetry"))
)
spark.sql(f"ALTER TABLE {cfg.table('sensor_telemetry')} CLUSTER BY (line_id, ts)")

spark.sql(f"""
CREATE OR REPLACE TABLE {cfg.table('fct_line_shift')} AS
SELECT
  line_id,
  date_sub(current_date(), 1) AS shift_date,
  'Morning' AS shift_name,
  28800 AS planned_seconds,
  CAST(20000 + (id % 5000) AS BIGINT) AS run_seconds,
  CAST(900 + (id % 120) AS BIGINT) AS units,
  CAST(850 + (id % 110) AS BIGINT) AS good_units,
  12.0 AS ideal_cycle_seconds
FROM range(24)
""")
tel_count = spark.table(cfg.table("sensor_telemetry")).count()
assert tel_count == n, (tel_count, n)
print("PASS telemetry", tel_count)
print("PASS fct_line_shift", spark.table(cfg.table("fct_line_shift")).count())
